# Decoder-Only Sprachmodelle: SFT vs. DPO Alignment Experiment

Dieses Notebook analysiert das **Decoder-Only Paradigma (Qwen-2.5-1.5B-Instruct)** auf dem ungesehenen **Lebenshilfe Gold-Standard Testset**:
1. **Reward-Modell & Stil-Einfachheit (	ext{style}$)** via vortrainiertem BiLSTM MixUp Regressor
2. **Semantische Quelltreue (	ext{sem}$)** & **Referenz-Ähnlichkeit ($	ext{Sim(Ref)}$)** via Sentence-BERT (mpnet)
3. **Lesbarkeitsindizes**: Flesch Reading Ease (DE), LIX Index, Wiener Sachtextformel (WSTF)
4. **Translations-Metriken**: BLEU & ROUGE-L gegen menschliche Gold-Referenz
5. **Linguistische Regeltreue**: Genitiv-Tilgung, Nominalstil-Vermeidung, Satzlänge, Passiv-Dichte

In [ ]:
import os
import sys
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "data")) and os.path.exists(os.path.join(p, "results")):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser("~/Documents/Master Thesis"))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Arbeitsverzeichnis:", os.getcwd())

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "figure.dpi": 150
})


## 1. Evaluationsdaten laden (Lebenshilfe Benchmark Testset)

In [ ]:
BENCHMARK_PATH = os.path.join(REPO_ROOT, "results/evaluation/benchmark_5way_decoder_vs_encoder_decoder.csv")
SUMMARY_PATH = os.path.join(REPO_ROOT, "results/evaluation/master_benchmark_summary.csv")

df_bench = pd.read_csv(BENCHMARK_PATH)
df_summary = pd.read_csv(SUMMARY_PATH)

print(f"Geladene Lebenshilfe Benchmark Testdaten: {len(df_bench)} Artikel")
print(f"Spalten im Benchmark-Datensatz: {list(df_bench.columns[:10])} ...")

# Regel-Adhärenz Helferfunktion für sprachliche Oberflächenanalyse
def compute_rule_metrics(text):
    words = [w for w in re.findall(r"\b\w+\b", str(text or "")) if w]
    sentences = [s.strip() for s in re.split(r"[.!?]+", str(text or "")) if s.strip()]
    num_words = len(words) or 1
    num_sents = len(sentences) or 1
    avg_sent_len = num_words / num_sents
    avg_word_len = sum(len(w) for w in words) / num_words
    long_words = sum(1 for w in words if len(w) > 6) / num_words
    genitives = sum(1 for w in words if re.search(r"(des|eines|unseres|ihres|dieses|[a-zäöü]+s)$", w, re.I)) / num_words
    nominals = sum(1 for w in words if re.search(r"(ung|keit|heit|schaft|ismus|tion|tät)$", w, re.I)) / num_words
    passives = sum(1 for s in sentences if re.search(r"\b(wird|werden|wurde|wurden|worden)\b", s, re.I)) / num_sents
    return {
        "avg_sent_len": avg_sent_len,
        "avg_word_len": avg_word_len,
        "long_words_ratio": long_words,
        "genitive_ratio": genitives,
        "nominal_ratio": nominals,
        "passive_ratio": passives
    }

# Regel-Metriken für SFT, DPO und AS Quelle berechnen
df_as_rules = pd.DataFrame([compute_rule_metrics(t) for t in df_bench["source_text"]])
df_sft_rules = pd.DataFrame([compute_rule_metrics(t) for t in df_bench["gen_dec_sft"]])
df_dpo_rules = pd.DataFrame([compute_rule_metrics(t) for t in df_bench["gen_dec_dpo"]])


## 2. Quantitative Metrik-Tabelle: SFT vs. DPO Alignment auf Lebenshilfe Benchmark

In [ ]:
metrics_config = [
    # Metrik-Key, Label, Ziel, higher_is_better
    ("r_style", "1. Simplicity Reward (BiLSTM R_style)", "Höher = Einfachere Sprache", True),
    ("r_sem_as", "2. SBERT-Quelltreue (AS Cosine Sim)", "Höher = Höhere Informationstreue", True),
    ("composite", "3. Composite Total Reward (0.5/0.5)", "Höher = Optimaler Trade-Off", True),
    ("flesch", "4. Flesch Reading Ease (DE)", "Höher = Bessere Lesbarkeit", True),
    ("lix", "5. LIX Lesbarkeitsindex", "Niedriger = Kürzere Sätze & Wörter", False),
    ("wiener", "6. Wiener Sachtextformel (WSTF)", "Niedriger = Geringere Schwierigkeitsstufe", False),
    ("bleu", "7. BLEU-4 Score (vs. Gold)", "Höher = N-Gramm Übereinstimmung mit Referenz", True),
    ("rouge_l", "8. ROUGE-L F1 (vs. Gold)", "Höher = Sequenztreue zur Gold-Referenz", True),
    ("rule_avg_sent_len", "9. Mittlere Satzlänge (Wörter)", "Niedriger = Kürzere Einzelsätze", False),
    ("rule_avg_word_len", "10. Mittlere Wortlänge (Buchstaben)", "Niedriger = Kürzere Wörter", False),
    ("rule_genitive_ratio", "11. Genitiv-Quote (pro Wort)", "Niedriger = Erfolgreiche Genitiv-Tilgung", False),
    ("rule_nominal_ratio", "12. Nominalstil-Quote (pro Wort)", "Niedriger = Verbalstil", False),
    ("rule_passive_ratio", "13. Passiv-Dichte (pro Satz)", "Niedriger = Aktiv-Konstruktionen", False)
]

rows = []
for key, name, desc, higher_better in metrics_config:
    if key.startswith("rule_"):
        r_key = key.replace("rule_", "")
        as_vals = df_as_rules[r_key]
        sft_vals = df_sft_rules[r_key]
        dpo_vals = df_dpo_rules[r_key]
    else:
        as_vals = pd.Series([np.nan] * len(df_bench))
        sft_vals = df_bench[f"{key}_dec_sft"]
        dpo_vals = df_bench[f"{key}_dec_dpo"]

    sft_m, sft_std = sft_vals.mean(), sft_vals.std()
    dpo_m, dpo_std = dpo_vals.mean(), dpo_vals.std()
    as_m = as_vals.mean() if not as_vals.isna().all() else np.nan

    delta = dpo_m - sft_m
    pct = (delta / abs(sft_m) * 100) if sft_m != 0 else 0

    t_stat, p_val = stats.ttest_rel(dpo_vals.dropna(), sft_vals.dropna())
    sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "n.s."))

    rows.append({
        "Metrik": name,
        "Ausgangstext (AS)": f"{as_m:.3f}" if not np.isnan(as_m) else "-",
        "Decoder-Only SFT": f"{sft_m:.3f} ± {sft_std:.3f}",
        "Decoder-Only DPO": f"{dpo_m:.3f} ± {dpo_std:.3f}",
        "Δ (DPO - SFT)": f"{delta:+.3f} ({pct:+.1f}%)",
        "Signifikanz (p)": f"{p_val:.4f} ({sig})",
        "Zielkriterium": desc
    })

df_comp = pd.DataFrame(rows)
display(df_comp)


## 3. Multi-Panel Visualisierung: Verteilungsanalyse von Reward, Quelltreue & Lesbarkeit

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

# Datensatz für Boxplots aufbereiten
df_plot_sft = pd.DataFrame({
    "Modell": "Decoder-Only SFT",
    "r_style": df_bench["r_style_dec_sft"],
    "r_sem_as": df_bench["r_sem_as_dec_sft"],
    "flesch": df_bench["flesch_dec_sft"],
    "lix": df_bench["lix_dec_sft"],
    "genitive": df_sft_rules["genitive_ratio"],
    "nominal": df_sft_rules["nominal_ratio"]
})

df_plot_dpo = pd.DataFrame({
    "Modell": "Decoder-Only DPO",
    "r_style": df_bench["r_style_dec_dpo"],
    "r_sem_as": df_bench["r_sem_as_dec_dpo"],
    "flesch": df_bench["flesch_dec_dpo"],
    "lix": df_bench["lix_dec_dpo"],
    "genitive": df_dpo_rules["genitive_ratio"],
    "nominal": df_dpo_rules["nominal_ratio"]
})

df_plot = pd.concat([df_plot_sft, df_plot_dpo], ignore_index=True)
palette = {"Decoder-Only SFT": "#3498db", "Decoder-Only DPO": "#e74c3c"}

plot_configs = [
    ("r_style", "(A) Style Reward Score (BiLSTM)", "Simplicity Score (0-1, Höher = Besser)"),
    ("r_sem_as", "(B) Semantische Quelltreue (SBERT zu AS)", "Cosine Similarity (Höher = Besser)"),
    ("flesch", "(C) Flesch Reading Ease (DE)", "Flesch Score (Höher = Leichter)"),
    ("lix", "(D) LIX Lesbarkeitsindex", "LIX Score (Niedriger = Kürzere Sätze)"),
    ("genitive", "(E) Genitiv-Quote", "Genitiv-Anteil (Niedriger = Besser)"),
    ("nominal", "(F) Nominalstil-Quote", "Nominalstil-Anteil (Niedriger = Besser)")
]

for idx, (col, title, ylabel) in enumerate(plot_configs):
    ax = axes[idx]
    sns.boxplot(data=df_plot, x="Modell", y=col, ax=ax, palette=palette, hue="Modell", legend=False,
                showmeans=True, meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 8})
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=10)
    ax.tick_params(axis="x", labelsize=11)

plt.suptitle("Decoder-Only Experiment (Qwen-2.5-1.5B): SFT vs. DPO Alignment auf Lebenshilfe Benchmark", fontsize=13, fontweight="bold")
plt.tight_layout()
plot_out = os.path.join(REPO_ROOT, "figures/decoder_only_sft_vs_dpo_comparison.png")
os.makedirs(os.path.dirname(plot_out), exist_ok=True)
plt.savefig(plot_out, dpi=300, bbox_inches="tight")
print(f"Plot gespeichert unter: {plot_out}")
plt.show()


## 4. Qualitatives Side-by-Side Audit auf ungesehenen Lebenshilfe Testartikeln

In [ ]:
print("=" * 110)
print("DETAIL-AUDIT: SIDE-BY-SIDE VERGLEICH DER DECODER-ONLY GENERIERUNGEN MIT REWARD- & QUELLTREUE-METRIKEN")
print("=" * 110)

def clean_snippet(text, max_len=300):
    t = str(text or "").strip()
    if t.endswith("..."):
        t = t[:-3].strip()
    if len(t) > max_len:
        return t[:max_len].strip() + " [...]"
    return t

for idx in range(min(5, len(df_bench))):
    as_t = clean_snippet(df_bench.iloc[idx]["source_text"])
    gold_t = clean_snippet(df_bench.iloc[idx]["target_text"])
    sft_t = clean_snippet(df_bench.iloc[idx]["gen_dec_sft"])
    dpo_t = clean_snippet(df_bench.iloc[idx]["gen_dec_dpo"])

    sft_r = df_bench.iloc[idx]["r_style_dec_sft"]
    sft_sem = df_bench.iloc[idx]["r_sem_as_dec_sft"]
    sft_fl = df_bench.iloc[idx]["flesch_dec_sft"]
    sft_lx = df_bench.iloc[idx]["lix_dec_sft"]

    dpo_r = df_bench.iloc[idx]["r_style_dec_dpo"]
    dpo_sem = df_bench.iloc[idx]["r_sem_as_dec_dpo"]
    dpo_fl = df_bench.iloc[idx]["flesch_dec_dpo"]
    dpo_lx = df_bench.iloc[idx]["lix_dec_dpo"]

    print()
    print(f"[BEISPIEL {idx+1}]")
    print(f"[AS-QUELLE]      : {as_t}")
    print(f"[GOLD-REFERENZ]  : {gold_t}")
    print()
    print(f"[SFT-OUTPUT]     : {sft_t}")
    print(f"  -> Metriken: Style-Reward: {sft_r:.3f} | SBERT(AS): {sft_sem:.3f} | Flesch: {sft_fl:.1f} | LIX: {sft_lx:.1f}")
    print()
    print(f"[DPO-OUTPUT]     : {dpo_t}")
    print(f"  -> Metriken: Style-Reward: {dpo_r:.3f} | SBERT(AS): {dpo_sem:.3f} | Flesch: {dpo_fl:.1f} | LIX: {dpo_lx:.1f}")
    print("-" * 110)
